# NeuroMotion — First Model Exploration (Milestone 4 preparation)

**Task:** classify recorded EEG windows as **Rest**, **Left-hand motor imagery**, or **Right-hand motor imagery**. The classifier is an exploratory model for replaying public recordings in a virtual interaction; it is not a medical device and does not control a real prosthesis.

This notebook documents the initial single-subject experiment, a controlled follow-up, and a separate all-subject, subject-independent cross-validation experiment. The initial run completed, but its validation predictions show that EEGNet predicted **Rest for every window**. Raw accuracy (0.50) only matches the training-majority baseline; balanced accuracy is 0.333 and macro-F1 is 0.222. The 0.60 and 0.70 confidence gates both rejected all 30 windows. These are genuine negative results, not a usable command model.

## Experiment design

- **AI role:** three-class EEG window classification.
- **Library / first candidate:** Braindecode `EEGClassifier` with the compact **EEGNet** architecture, trained from scratch using PyTorch.
- **Alternative candidates:** Braindecode `ShallowFBCSPNet` and a non-deep-learning CSP + LDA baseline remain future comparisons, not measured results.
- **Dataset:** PhysioNet EEG Motor Movement/Imagery Dataset (EEGMMIDB) v1.0.0.
- **Preprocessing:** EEG channels only; 4–38 Hz band-pass; 3-second windows from 0.5 to 3.5 seconds after `T0`, `T1`, or `T2`; convert V to microvolts; standardize each channel using mean and standard deviation fit **on training data only**.
- **Original single-subject split:** S001 R04 for training, R08 for development/validation, and R12 reserved for a final test of that narrow within-subject experiment. This is not a cross-subject result.
- **Added all-subject experiment:** attempt to include all 109 PhysioNet subjects using imagined unilateral-fist runs R04/R08/R12. Use five-fold subject-grouped cross-validation: every eligible subject is held out for test exactly once and is included in training in the other folds. No subject may appear in both training and test within a fold. Missing or unusable subjects/runs are reported rather than silently counted as included.
- **Initial configuration:** 10 epochs, batch size 8, Adam learning rate 0.001.
- **Gate diagnostic:** compare 0.60 and 0.70 using the same initial-model probabilities. Because no windows passed either gate, this comparison does **not** identify a useful threshold.
- **Controlled training follow-up:** test one change—class-weighted cross-entropy—to address the initial model's collapse to the majority class. Keep architecture, seed, epochs, optimizer, batch size, train/validation split, and preprocessing fixed. The new comparison cells must be run to generate results; no follow-up result is fabricated here.

**Submission rule:** run every cell locally and save the notebook with outputs visible. Report the failed baseline honestly. The all-subject cross-validation is a separate experiment: it uses R04/R08/R12 across subject-disjoint folds and reports out-of-fold predictions for every eligible subject. The original S001-only R12 reservation remains specific to the original within-subject experiment; do not mix the two result tables.


## Step 1 — define the role of AI and first candidate

The learned component maps a preprocessed EEG segment to one of three learned labels. A deterministic rule outside the model turns a low-confidence prediction into `NO COMMAND`. A confident `Rest` prediction is a valid class and means no movement; it is not the same as a rejected low-confidence result. The live replay layer must never execute a movement after `NO COMMAND`.

| Candidate | Why consider it | Decision for this experiment |
|---|---|---|
| Braindecode EEGNet | Compact EEG-specific temporal and spatial convolutional model; documented Braindecode training API | **Selected** for the first end-to-end neural model |
| Braindecode ShallowFBCSPNet | A second EEG model designed around temporal filtering and spatial features; useful for motor imagery | Future comparison after the pipeline works |
| CSP + LDA | Classical and relatively data-efficient motor-imagery baseline | Future benchmark; do not call it measured yet |

EEGNet is not pretrained here: its weights are initialized and fitted to the training data. The model is a starting point, not a claim that deep learning will outperform the classical baseline on this small within-subject split.

## Step 2 — environment and reproducibility

Use a dedicated Python 3.11 environment. This avoids mixing the project dependencies with the Python 3.14 environment where plotting/native-library issues were reported earlier. This notebook deliberately uses tables and printed diagnostics rather than Matplotlib plots.

In [2]:
import json
import random
import time
import platform
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import mne
import sklearn
import braindecode
from braindecode import EEGClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("MNE:", mne.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("Braindecode:", braindecode.__version__)
print("Device:", DEVICE)

Python: 3.12.15
MNE: 1.13.2
scikit-learn: 1.9.1
PyTorch: 2.14.1+cu130
Braindecode: 1.8.1
Device: cuda


## Step 2 — configure and verify the local dataset path

Keep raw data outside Git. Point `DATA_ROOT` to the folder containing subject folders such as `S001/S001R04.edf`. Common locations are searched as a convenience, but the notebook never downloads data silently.

In [3]:
SUBJECT = "S001"
TRAIN_RUNS = ("R04",)
VALIDATION_RUNS = ("R08",)
RESERVED_TEST_RUNS = ("R12",)  # Quality/label/hash audit only; do not use for fitting or tuning.

# Change this to the parent folder containing S001, S002, ... if necessary.
DATA_ROOT = Path("data")
SEARCH_ROOTS = [
    DATA_ROOT,
    Path("eegmmidb"),
    Path("eegmmidb/1.0.0"),
    Path.home() / "mne_data" / "MNE-eegbci-data" / "files" / "eegmmidb" / "1.0.0",
    Path.home() / "Downloads" / "eegmmidb" / "1.0.0",
]

# Use the first candidate tree that contains the required training recording.
for root in SEARCH_ROOTS:
    if root.exists() and any(root.rglob(f"{SUBJECT}{TRAIN_RUNS[0]}.edf")):
        DATA_ROOT = root
        break


def find_recording(subject: str, run: str) -> Path:
    filename = f"{subject}{run}.edf"
    matches = sorted(DATA_ROOT.rglob(filename)) if DATA_ROOT.exists() else []
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Set DATA_ROOT to the folder containing "
            f"the subject folders. Current DATA_ROOT: {DATA_ROOT.resolve()}"
        )
    if len(matches) > 1:
        print(f"Warning: found {len(matches)} copies of {filename}; using {matches[0]}")
    return matches[0]

required_files = {
    run: find_recording(SUBJECT, run)
    for run in (*TRAIN_RUNS, *VALIDATION_RUNS)
}
print("Dataset root:", DATA_ROOT.resolve())
print("Subject:", SUBJECT)
for run, path in required_files.items():
    print(f"{run}: {path}")
print("Reserved final-test run(s) not loaded:", ", ".join(RESERVED_TEST_RUNS))

Dataset root: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\data
Subject: S001
R04: data\S001\S001R04.edf
R08: data\S001\S001R08.edf
Reserved final-test run(s) not loaded: R12


## Step 2 — load, preprocess, epoch, and check labels

PhysioNet's `T0`, `T1`, and `T2` meanings depend on run type. These are the unilateral **imagined** fist runs: `T0 = Rest`, `T1 = Left Hand imagery`, and `T2 = Right Hand imagery`. Each run is loaded and epoched independently. Any missing event type, non-finite samples, unexpected sampling rate, or channel mismatch raises an error rather than being silently repaired.

The epoch is 0.5–3.5 seconds after each event onset. Events that do not have enough following data may be dropped by MNE; the notebook reports requested versus retained windows.

In [4]:
EVENT_TO_LABEL = {"T0": 0, "T1": 1, "T2": 2}
CLASS_NAMES = ["Rest", "Left Hand", "Right Hand"]
CLASS_TO_LABEL = {name: idx for idx, name in enumerate(CLASS_NAMES)}

LOW_CUT_HZ = 4.0
HIGH_CUT_HZ = 38.0
WINDOW_TMIN = 0.5
WINDOW_TMAX = 3.5
EXPECTED_SFREQ = 160.0


def load_run(subject: str, run: str):
    path = find_recording(subject, run)
    raw = mne.io.read_raw_edf(path, preload=True, verbose="ERROR")
    sfreq = float(raw.info["sfreq"])
    if not np.isclose(sfreq, EXPECTED_SFREQ):
        raise ValueError(f"Unexpected sampling rate {sfreq} Hz in {path}; expected {EXPECTED_SFREQ} Hz.")

    raw.pick(picks="eeg")
    if len(raw.ch_names) == 0:
        raise ValueError(f"No EEG channels were found in {path}.")
    channel_names = list(raw.ch_names)
    raw.filter(l_freq=LOW_CUT_HZ, h_freq=HIGH_CUT_HZ, picks="eeg", verbose="ERROR")

    events, found_event_id = mne.events_from_annotations(
        raw, regexp=r"^(T0|T1|T2)$", verbose="ERROR"
    )
    missing = sorted(set(EVENT_TO_LABEL) - set(found_event_id))
    if missing:
        raise ValueError(f"Missing expected annotations {missing} in {path}; found {found_event_id}.")
    selected_event_id = {name: found_event_id[name] for name in EVENT_TO_LABEL}

    epochs = mne.Epochs(
        raw,
        events,
        event_id=selected_event_id,
        tmin=WINDOW_TMIN,
        tmax=WINDOW_TMAX,
        baseline=None,
        picks="eeg",
        preload=True,
        reject_by_annotation=True,
        verbose="ERROR",
    )
    X = epochs.get_data().astype(np.float32) * 1e6  # MNE stores EEG in volts; model uses microvolts.
    if not np.isfinite(X).all():
        raise ValueError(f"Non-finite EEG values found in {path}; inspect quality before training.")

    code_to_label = {selected_event_id[event_name]: EVENT_TO_LABEL[event_name]
                     for event_name in EVENT_TO_LABEL}
    code_to_event = {selected_event_id[event_name]: event_name
                     for event_name in EVENT_TO_LABEL}
    y = np.asarray([code_to_label[int(code)] for code in epochs.events[:, 2]], dtype=np.int64)
    onsets = (epochs.events[:, 0] - raw.first_samp) / sfreq
    metadata = pd.DataFrame({
        "subject": subject,
        "run": run,
        "event_onset_sec": np.round(onsets, 3),
        "annotation": [code_to_event[int(code)] for code in epochs.events[:, 2]],
        "label_id": y,
        "true_class": [CLASS_NAMES[int(value)] for value in y],
    })
    report = {
        "subject": subject,
        "run": run,
        "file": str(path),
        "sampling_rate_hz": sfreq,
        "channels": len(channel_names),
        "requested_events_T0_T1_T2": len(events),
        "retained_epochs": len(epochs),
        "dropped_epochs": len(events) - len(epochs),
        "samples_per_epoch": int(X.shape[2]) if len(X) else 0,
        "finite_values": bool(np.isfinite(X).all()),
        "min_uV": float(np.min(X)) if X.size else np.nan,
        "max_uV": float(np.max(X)) if X.size else np.nan,
    }
    return X, y, metadata, channel_names, sfreq, report

run_data = {}
run_reports = []
for run in (*TRAIN_RUNS, *VALIDATION_RUNS):
    loaded = load_run(SUBJECT, run)
    run_data[run] = loaded
    run_reports.append(loaded[-1])

quality_summary = pd.DataFrame(run_reports)
display(quality_summary)

reference_channels = run_data[TRAIN_RUNS[0]][3]
for run in (*TRAIN_RUNS, *VALIDATION_RUNS):
    if run_data[run][3] != reference_channels:
        raise ValueError(f"Channel names/order differ in {run}. Harmonize channels explicitly before training.")
    if not np.isclose(run_data[run][4], EXPECTED_SFREQ):
        raise ValueError(f"Unexpected sampling rate in {run}.")

X_train_raw = np.concatenate([run_data[run][0] for run in TRAIN_RUNS], axis=0)
y_train = np.concatenate([run_data[run][1] for run in TRAIN_RUNS], axis=0)
meta_train = pd.concat([run_data[run][2] for run in TRAIN_RUNS], ignore_index=True)
X_val_raw = np.concatenate([run_data[run][0] for run in VALIDATION_RUNS], axis=0)
y_val = np.concatenate([run_data[run][1] for run in VALIDATION_RUNS], axis=0)
meta_val = pd.concat([run_data[run][2] for run in VALIDATION_RUNS], ignore_index=True)

if len(X_train_raw) == 0 or len(X_val_raw) == 0:
    raise ValueError("No training or validation windows were retained. Check annotations/window timing.")
if set(np.unique(y_train)) != set(range(len(CLASS_NAMES))):
    raise ValueError(f"Training split does not contain all three classes: {Counter(y_train)}")
if not set(np.unique(y_val)).issubset(set(range(len(CLASS_NAMES)))):
    raise ValueError("Unexpected validation labels.")

print("Train tensor shape (windows, channels, time):", X_train_raw.shape)
print("Validation tensor shape (windows, channels, time):", X_val_raw.shape)
print("Training class counts:")
display(pd.Series([CLASS_NAMES[int(v)] for v in y_train]).value_counts().reindex(CLASS_NAMES, fill_value=0).rename("count").to_frame())
print("Validation class counts:")
display(pd.Series([CLASS_NAMES[int(v)] for v in y_val]).value_counts().reindex(CLASS_NAMES, fill_value=0).rename("count").to_frame())

,subject,run,file,sampling_rate_hz,channels,requested_events_T0_T1_T2,retained_epochs,dropped_epochs,samples_per_epoch,finite_values,min_uV,max_uV
0,S001,R04,data\S001\S001R04.edf,160.0,64,30,30,0,481,True,-315.934937,305.064911
1,S001,R08,data\S001\S001R08.edf,160.0,64,30,30,0,481,True,-251.535553,296.216949


Train tensor shape (windows, channels, time): (30, 64, 481)
Validation tensor shape (windows, channels, time): (30, 64, 481)
Training class counts:


,count
Rest,15
Left Hand,8
Right Hand,7


Validation class counts:


,count
Rest,15
Left Hand,8
Right Hand,7


## Step 2 — fit normalization only on training data

This is the only learned preprocessing step in this notebook. Per-channel means and standard deviations are calculated from the training run only, then reused unchanged on validation. This helps keep the validation data from influencing model preparation.

In [5]:
channel_mean = X_train_raw.mean(axis=(0, 2), keepdims=True)
channel_std = X_train_raw.std(axis=(0, 2), keepdims=True)
channel_std = np.where(channel_std < 1e-6, 1.0, channel_std)

X_train = ((X_train_raw - channel_mean) / channel_std).astype(np.float32)
X_val = ((X_val_raw - channel_mean) / channel_std).astype(np.float32)

assert X_train.shape[1:] == X_val.shape[1:]
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()

print("Normalization fitted on training windows only.")
print("Training normalized mean/std:", round(float(X_train.mean()), 4), round(float(X_train.std()), 4))
print("Validation normalized mean/std (not forced to 0/1):", round(float(X_val.mean()), 4), round(float(X_val.std()), 4))

Normalization fitted on training windows only.
Training normalized mean/std: 0.0 1.0
Validation normalized mean/std (not forced to 0/1): -0.0028 0.9418


## Step 3 — fit the first candidate end-to-end

The run uses a fixed seed, EEGNet, Adam, learning rate 0.001, batch size 8, and 10 epochs. Braindecode's `EEGClassifier` is given `train_split=None` because the run-level validation set is already separate; an internal random split of the training run is unnecessary for this first exploration.

In [6]:
INITIAL_CONFIG = {
    "model": "Braindecode EEGNet",
    "max_epochs": 10,
    "batch_size": 8,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "random_seed": SEED,
    "confidence_threshold_initial": 0.60,
    "confidence_threshold_revised": 0.70,
}

net = EEGClassifier(
    "EEGNet",
    module__n_chans=int(X_train.shape[1]),
    module__n_outputs=len(CLASS_NAMES),
    module__n_times=int(X_train.shape[2]),
    module__sfreq=float(EXPECTED_SFREQ),
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.Adam,
    lr=INITIAL_CONFIG["learning_rate"],
    max_epochs=INITIAL_CONFIG["max_epochs"],
    batch_size=INITIAL_CONFIG["batch_size"],
    train_split=None,
    iterator_train__shuffle=True,
    iterator_train__drop_last=False,
    device=DEVICE,
    verbose=1,
)

training_start = time.perf_counter()
net.fit(X_train, y_train)
training_seconds = time.perf_counter() - training_start
print(f"Completed initial training in {training_seconds:.2f} seconds on {DEVICE}.")

# Save the epoch history for the report. These are training logs, not validation scores.
OUTPUT_DIR = Path("outputs")
ARTIFACT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
history_df = pd.DataFrame(net.history.to_list())
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
display(history_df.tail(5))

  epoch    train_loss     dur
-------  ------------  ------
      1        1.0754  1.0926
      2        0.9996  0.0150
      3        0.9642  0.0160
      4        0.9554  0.0160
      5        0.9372  0.0150
      6        0.9088  0.0150
      7        0.8817  0.0160
      8        0.8762  0.0150
      9        0.8728  0.0160
     10        0.8263  0.0150
Completed initial training in 2.55 seconds on cuda.


,batches,epoch,train_batch_count,dur,train_loss,train_loss_best
5,"[{'train_loss': 0.979208767414093, 'train_batc...",6,4,0.015000,0.908812,True
6,"[{'train_loss': 0.9331609010696411, 'train_bat...",7,4,0.016000,0.881678,True
7,"[{'train_loss': 0.9035722017288208, 'train_bat...",8,4,0.014997,0.876245,True
8,"[{'train_loss': 0.8011072278022766, 'train_bat...",9,4,0.016000,0.872753,True
9,"[{'train_loss': 1.0406920909881592, 'train_bat...",10,4,0.014999,0.826321,True


## Step 4 — evaluate the initial model on the validation run

Report raw classification metrics before confidence gating, including class-balanced metrics because accuracy alone can hide a weak class. The simple majority-class baseline is calculated from training labels only and predicts that one class for all validation windows. These metrics are preliminary because this first split covers one subject and one training run.

In [7]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

probabilities = np.asarray(net.predict_proba(X_val), dtype=float)
if probabilities.shape != (len(y_val), len(CLASS_NAMES)):
    raise ValueError(f"Unexpected probability matrix shape: {probabilities.shape}")
if not np.isfinite(probabilities).all():
    raise ValueError("Model produced non-finite probabilities.")
if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-3):
    raise ValueError("predict_proba rows do not sum to approximately 1; inspect Braindecode/skorch configuration.")

y_pred = probabilities.argmax(axis=1)
confidence = probabilities.max(axis=1)

model_raw_metrics = {
    "model": "EEGNet (raw prediction)",
    "validation_windows": int(len(y_val)),
    "accuracy": float(accuracy_score(y_val, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_val, y_pred)),
    "macro_f1": float(f1_score(y_val, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
}
majority_class = int(np.bincount(y_train, minlength=len(CLASS_NAMES)).argmax())
majority_pred = np.full_like(y_val, majority_class)
majority_metrics = {
    "model": f"Training majority baseline ({CLASS_NAMES[majority_class]})",
    "validation_windows": int(len(y_val)),
    "accuracy": float(accuracy_score(y_val, majority_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_val, majority_pred)),
    "macro_f1": float(f1_score(y_val, majority_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
}
raw_metric_table = pd.DataFrame([majority_metrics, model_raw_metrics])
display(raw_metric_table.round(4))

print("Classification report (validation only):")
print(classification_report(
    y_val, y_pred,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    zero_division=0,
))

confusion_df = pd.DataFrame(
    confusion_matrix(y_val, y_pred, labels=[0, 1, 2]),
    index=[f"true_{name}" for name in CLASS_NAMES],
    columns=[f"pred_{name}" for name in CLASS_NAMES],
)
print("Confusion matrix (counts):")
display(confusion_df)
raw_metric_table.to_csv(OUTPUT_DIR / "raw_validation_metrics.csv", index=False)
confusion_df.to_csv(OUTPUT_DIR / "raw_confusion_matrix.csv")

,model,validation_windows,accuracy,balanced_accuracy,macro_f1
0,Training majority baseline (Rest),30,0.5,0.3333,0.2222
1,EEGNet (raw prediction),30,0.5,0.3333,0.2222


Classification report (validation only):
              precision    recall  f1-score   support

        Rest       0.50      1.00      0.67        15
   Left Hand       0.00      0.00      0.00         8
  Right Hand       0.00      0.00      0.00         7

    accuracy                           0.50        30
   macro avg       0.17      0.33      0.22        30
weighted avg       0.25      0.50      0.33        30

Confusion matrix (counts):


,pred_Rest,pred_Left Hand,pred_Right Hand
true_Rest,15,0,0
true_Left Hand,8,0,0
true_Right Hand,7,0,0


## Step 5 — inspect weak outputs and quantify confidence-gate behavior

`Rest` remains a valid label and produces no movement. If the largest predicted class probability is below a threshold, the runtime emits `NO COMMAND` and must stop. A confidently wrong classification is not knowable from confidence alone at runtime; the validation labels let us count it offline.

The two threshold rows below use the **same trained model, same validation windows, and same probabilities**. Only the confidence threshold changes from 0.60 to 0.70.

In [8]:
def gate_metrics(threshold: float) -> dict:
    accepted = confidence >= threshold
    emitted_movement = accepted & np.isin(y_pred, [1, 2])
    true_rest = y_val == 0
    false_movement_on_rest = emitted_movement & true_rest
    wrong_movement = emitted_movement & (y_pred != y_val)

    return {
        "configuration": f"EEGNet + confidence gate {threshold:.2f}",
        "confidence_threshold": threshold,
        "validation_windows": int(len(y_val)),
        "raw_accuracy": float(accuracy_score(y_val, y_pred)),
        "raw_macro_f1": float(f1_score(y_val, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "accepted_output_rate": float(accepted.mean()),
        "no_command_rate": float((~accepted).mean()),
        "movement_command_rate": float(emitted_movement.mean()),
        "false_movement_on_rest_rate": (
            float(false_movement_on_rest.sum() / true_rest.sum()) if true_rest.any() else np.nan
        ),
        "wrong_movement_command_rate": (
            float(wrong_movement.sum() / emitted_movement.sum()) if emitted_movement.any() else np.nan
        ),
        "movement_commands": int(emitted_movement.sum()),
        "wrong_movement_commands": int(wrong_movement.sum()),
    }

experiment_results = pd.DataFrame([
    gate_metrics(INITIAL_CONFIG["confidence_threshold_initial"]),
    gate_metrics(INITIAL_CONFIG["confidence_threshold_revised"]),
])
display(experiment_results.round(4))
experiment_results.to_csv(OUTPUT_DIR / "experiment_results.csv", index=False)

# Make individual validation outcomes auditable; include every wrong prediction and low-confidence case.
inspection = meta_val.copy().reset_index(drop=True)
inspection["true_class"] = [CLASS_NAMES[int(v)] for v in y_val]
inspection["predicted_class"] = [CLASS_NAMES[int(v)] for v in y_pred]
inspection["confidence"] = confidence
inspection["correct_raw_prediction"] = (y_val == y_pred)
inspection["threshold_060_output"] = [
    "NO COMMAND" if p < INITIAL_CONFIG["confidence_threshold_initial"]
    else ("REST (no movement)" if cls == 0 else CLASS_NAMES[int(cls)])
    for p, cls in zip(confidence, y_pred)
]
inspection["threshold_070_output"] = [
    "NO COMMAND" if p < INITIAL_CONFIG["confidence_threshold_revised"]
    else ("REST (no movement)" if cls == 0 else CLASS_NAMES[int(cls)])
    for p, cls in zip(confidence, y_pred)
]
inspection["review_status_at_060"] = [
    "Low confidence - rejected" if p < INITIAL_CONFIG["confidence_threshold_initial"]
    else ("Confident misclassification" if true != pred else "Correct accepted output")
    for p, true, pred in zip(confidence, y_val, y_pred)
]
inspection.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)

needs_review = inspection[inspection["review_status_at_060"] != "Correct accepted output"]
if needs_review.empty:
    print("No incorrect or rejected windows at threshold 0.60; displaying the 10 least-confident examples instead.")
    review_rows = inspection.nsmallest(min(10, len(inspection)), "confidence")
else:
    review_rows = needs_review.sort_values(
        ["correct_raw_prediction", "confidence"], ascending=[True, True]
    ).head(10)
print(f"Unique validation windows inspected: {len(inspection)}; rows shown: {len(review_rows)}")
display(review_rows[[
    "run", "event_onset_sec", "true_class", "predicted_class", "confidence",
    "threshold_060_output", "threshold_070_output", "review_status_at_060"
]])

print("Wrong raw classifications:", int((y_val != y_pred).sum()), "of", int(len(y_val)))
print("Low-confidence rejections at 0.60:", int((confidence < 0.60).sum()), "of", int(len(y_val)))

,configuration,confidence_threshold,validation_windows,raw_accuracy,raw_macro_f1,accepted_output_rate,no_command_rate,movement_command_rate,false_movement_on_rest_rate,wrong_movement_command_rate,movement_commands,wrong_movement_commands
0,EEGNet + confidence gate 0.60,0.6,30,0.5,0.2222,0.0,1.0,0.0,0.0,NaN,0,0
1,EEGNet + confidence gate 0.70,0.7,30,0.5,0.2222,0.0,1.0,0.0,0.0,NaN,0,0


Unique validation windows inspected: 30; rows shown: 10


,run,event_onset_sec,true_class,predicted_class,confidence,threshold_060_output,threshold_070_output,review_status_at_060
1,R08,4.2,Left Hand,Rest,0.341435,NO COMMAND,NO COMMAND,Low confidence - rejected
25,R08,103.8,Right Hand,Rest,0.344778,NO COMMAND,NO COMMAND,Low confidence - rejected
19,R08,78.9,Left Hand,Rest,0.349486,NO COMMAND,NO COMMAND,Low confidence - rejected
9,R08,37.4,Left Hand,Rest,0.351597,NO COMMAND,NO COMMAND,Low confidence - rejected
3,R08,12.5,Right Hand,Rest,0.353253,NO COMMAND,NO COMMAND,Low confidence - rejected
21,R08,87.2,Left Hand,Rest,0.353916,NO COMMAND,NO COMMAND,Low confidence - rejected
29,R08,120.4,Left Hand,Rest,0.353922,NO COMMAND,NO COMMAND,Low confidence - rejected
11,R08,45.7,Right Hand,Rest,0.354531,NO COMMAND,NO COMMAND,Low confidence - rejected
27,R08,112.1,Left Hand,Rest,0.357949,NO COMMAND,NO COMMAND,Low confidence - rejected
5,R08,20.8,Left Hand,Rest,0.361016,NO COMMAND,NO COMMAND,Low confidence - rejected


Wrong raw classifications: 15 of 30
Low-confidence rejections at 0.60: 30 of 30


## Step 5 — measure one practical constraint: inference latency

This records CPU/GPU, number of unique validation windows, and single-window inference latency. To get a minimally useful timing distribution when the validation run is small, the timing probe repeats calls over existing validation windows until 100 calls have been timed. **Repeated calls are only for timing; accuracy and other model metrics above count each unique validation window once.** This is a preliminary local benchmark, not a claim of real-time system latency under every deployment condition.

In [9]:
# Synchronized model-only GPU inference benchmark.
# This is not the full replay latency: EDF I/O, filtering, gate, logging and UI are excluded.
latency_calls = max(100, len(X_val))
latency_ms = []

# Warm up model/device initialization before timing.
for i in range(min(10, len(X_val))):
    _ = net.predict_proba(X_val[[i]])
if DEVICE == "cuda":
    torch.cuda.synchronize()

for i in range(latency_calls):
    x_one = X_val[[i % len(X_val)]]
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = net.predict_proba(x_one)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    latency_ms.append((time.perf_counter() - t0) * 1000.0)

latency_summary = {
    "device": DEVICE,
    "unique_validation_windows": int(len(X_val)),
    "timed_single_window_calls": int(latency_calls),
    "median_single_window_inference_ms": float(np.median(latency_ms)),
    "p95_single_window_inference_ms": float(np.percentile(latency_ms, 95)),
    "measurement_note": (
        "CUDA synchronized before and after each measured call; calls repeat validation "
        "windows if fewer than 100 unique windows are available. Model-only inference; "
        "not end-to-end replay latency."
    ),
}
display(pd.DataFrame([latency_summary]).round(3))
with open(OUTPUT_DIR / "latency_summary_synchronized.json", "w", encoding="utf-8") as f:
    json.dump(latency_summary, f, indent=2)


,device,unique_validation_windows,timed_single_window_calls,median_single_window_inference_ms,p95_single_window_inference_ms,measurement_note
0,cuda,30,100,0.953,1.217,CUDA synchronized before and after each measur...


## Step 6 — interpret the confidence-gate diagnostic and save the initial handoff

The 0.60-versus-0.70 threshold comparison is an operational diagnostic, not a successful model improvement. In the saved initial run, all 30 validation windows fell below 0.60, so both thresholds produced `NO COMMAND`. This safely prevents movement, but it does not show that 0.70 is better; the model has not learned usable class separation yet.

The initial model handoff saves weights plus model dimensions, ordered channel names, filter/window parameters, class map, run IDs, and training-only normalization values. A future runtime must use the same ordered channels and exactly the same preprocessing/normalization. Do not deploy the initial weights as a usable controller while its validation output is collapsed to Rest/NO COMMAND.


In [10]:
weights_path = ARTIFACT_DIR / "neuromotion_eegnet_state_dict.pt"
config_path = ARTIFACT_DIR / "neuromotion_eegnet_config.json"
state_dict_cpu = {name: tensor.detach().cpu() for name, tensor in net.module_.state_dict().items()}
torch.save(state_dict_cpu, weights_path)

model_handoff = {
    "project": "NeuroMotion",
    "model_class": "braindecode.models.EEGNet",
    "classifier_wrapper": "braindecode.EEGClassifier",
    "braindecode_version": braindecode.__version__,
    "pytorch_version": torch.__version__,
    "subject": SUBJECT,
    "training_runs": list(TRAIN_RUNS),
    "validation_runs": list(VALIDATION_RUNS),
    "reserved_test_runs_not_used": list(RESERVED_TEST_RUNS),
    "class_mapping": {str(i): name for i, name in enumerate(CLASS_NAMES)},
    "channel_names_in_order": reference_channels,
    "sampling_rate_hz": EXPECTED_SFREQ,
    "bandpass_hz": [LOW_CUT_HZ, HIGH_CUT_HZ],
    "epoch_seconds_after_event": [WINDOW_TMIN, WINDOW_TMAX],
    "n_channels": int(X_train.shape[1]),
    "n_times": int(X_train.shape[2]),
    "input_units_before_standardization": "microvolts",
    "normalization": {
        "method": "per-channel z-score",
        "fit_split": "training run(s) only",
        "channel_mean_microvolts": channel_mean.reshape(-1).astype(float).tolist(),
        "channel_std_microvolts": channel_std.reshape(-1).astype(float).tolist(),
    },
    "training_configuration": INITIAL_CONFIG,
    "active_threshold_candidate": INITIAL_CONFIG["confidence_threshold_initial"],
    "revised_threshold_candidate_for_validation_comparison": INITIAL_CONFIG["confidence_threshold_revised"],
    "validation_raw_accuracy": model_raw_metrics["accuracy"],
    "validation_balanced_accuracy": model_raw_metrics["balanced_accuracy"],
    "validation_macro_f1": model_raw_metrics["macro_f1"],
    "notes": [
        "Exploratory within-subject model; not a medical device.",
        "Threshold selection remains a development decision, not a safety guarantee.",
        "Do not use the reserved R12 test run until model and threshold choices are frozen.",
        "Low confidence must lead to NO COMMAND and STOP; never execute the rejected prediction.",
    ],
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_handoff, f, indent=2)

print("Saved weights:", weights_path.resolve(), f"({weights_path.stat().st_size:,} bytes)")
print("Saved model config:", config_path.resolve())
print("Saved CSV/JSON outputs in:", OUTPUT_DIR.resolve())

Saved weights: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_state_dict.pt (19,318 bytes)
Saved model config: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_config.json
Saved CSV/JSON outputs in: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\outputs


## Automatically generated initial-experiment summary

The printed values and table below summarize the initial EEGNet run and its gate diagnostic. Interpret the result as a failed first baseline: accuracy matches the majority-class baseline, balanced accuracy equals the three-class chance reference (approximately 0.333), macro-F1 is low, and all validation outputs were rejected by both thresholds. The follow-up below tests class-weighted loss as a single targeted training change and compares both models on the same held-out validation run.


In [11]:
initial = experiment_results.iloc[0]
revised = experiment_results.iloc[1]

def pct_or_na(value, reason="not defined"):
    return reason if pd.isna(value) else f"{value:.1%}"

print(f"Initial model: EEGNet, {INITIAL_CONFIG['max_epochs']} epochs, Adam lr={INITIAL_CONFIG['learning_rate']}, batch={INITIAL_CONFIG['batch_size']}.")
print(f"Validation windows: {len(y_val)} from run(s) {', '.join(VALIDATION_RUNS)}; R12 remains reserved.")
print(f"Raw accuracy={model_raw_metrics['accuracy']:.3f}; balanced accuracy={model_raw_metrics['balanced_accuracy']:.3f}; macro-F1={model_raw_metrics['macro_f1']:.3f}.")
for label, row in [("0.60", initial), ("0.70", revised)]:
    print(
        f"Gate {label}: accepted={row['accepted_output_rate']:.1%}; "
        f"no-command={row['no_command_rate']:.1%}; "
        f"false movement on true-rest windows={row['false_movement_on_rest_rate']:.1%}; "
        f"wrong movement among emitted movement commands="
        f"{pct_or_na(row['wrong_movement_command_rate'], 'N/A (zero movement commands)')} ."
    )
print("Interpretation: both thresholds reject all validation windows, so the gate is safe in this sample but provides zero usable commands. Do not describe the undefined wrong-command rate as 0%.")


Initial model: EEGNet, 10 epochs, Adam lr=0.001, batch=8.
Validation windows: 30 from run(s) R08; R12 remains reserved.
Raw accuracy=0.500; balanced accuracy=0.333; macro-F1=0.222.
Gate 0.60: accepted=0.0%; no-command=100.0%; false movement on true-rest windows=0.0%; wrong movement among emitted movement commands=N/A (zero movement commands) .
Gate 0.70: accepted=0.0%; no-command=100.0%; false movement on true-rest windows=0.0%; wrong movement among emitted movement commands=N/A (zero movement commands) .
Interpretation: both thresholds reject all validation windows, so the gate is safe in this sample but provides zero usable commands. Do not describe the undefined wrong-command rate as 0%.


## Controlled follow-up — test class-weighted loss

### Why this change?

The saved baseline predicts `Rest` for all validation windows. Raising the confidence threshold cannot fix that failure; it only withholds more outputs. This follow-up tests **one training change only**: give each class a loss weight inversely proportional to its frequency in the training run. Both comparison models are re-created with the same seed and trained with the same EEGNet architecture, 10 epochs, optimizer, batch size, preprocessing, and whole-run split. The sole intended difference is the loss weights.

This is a hypothesis to test, not a guaranteed fix. With only 30 training windows, class weighting may or may not improve generalization. Use the same 30 R08 windows for both models, report the complete table/confusion matrices, and do not inspect reserved R12 while choosing a configuration.


In [12]:
# Construct a paired comparison so the only intended change is class-weighted loss.
class_counts = np.bincount(y_train, minlength=len(CLASS_NAMES)).astype(np.float64)
if np.any(class_counts == 0):
    raise ValueError(f"Cannot compute class weights; a training class is missing: {class_counts}")
class_weight_values = len(y_train) / (len(CLASS_NAMES) * class_counts)
class_weight_tensor = torch.as_tensor(class_weight_values, dtype=torch.float32)

print("Training counts by class:", {CLASS_NAMES[i]: int(class_counts[i]) for i in range(len(CLASS_NAMES))})
print("Class weights:", {CLASS_NAMES[i]: round(float(class_weight_values[i]), 4) for i in range(len(CLASS_NAMES))})


def fit_comparison_eegnet(loss_weights=None):
    """Fit EEGNet with the same seed/settings; optionally weight CE loss by class."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    extra_criterion_kwargs = {}
    if loss_weights is not None:
        # Skorch passes criterion__weight to torch.nn.CrossEntropyLoss.
        extra_criterion_kwargs["criterion__weight"] = torch.as_tensor(
            loss_weights, dtype=torch.float32
        )

    model = EEGClassifier(
        "EEGNet",
        module__n_chans=int(X_train.shape[1]),
        module__n_outputs=len(CLASS_NAMES),
        module__n_times=int(X_train.shape[2]),
        module__sfreq=float(EXPECTED_SFREQ),
        criterion=torch.nn.CrossEntropyLoss,
        optimizer=torch.optim.Adam,
        lr=INITIAL_CONFIG["learning_rate"],
        max_epochs=INITIAL_CONFIG["max_epochs"],
        batch_size=INITIAL_CONFIG["batch_size"],
        train_split=None,
        iterator_train__shuffle=True,
        iterator_train__drop_last=False,
        device=DEVICE,
        verbose=0,
        **extra_criterion_kwargs,
    )
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    return model, elapsed


paired_unweighted, paired_unweighted_seconds = fit_comparison_eegnet(loss_weights=None)
paired_weighted, paired_weighted_seconds = fit_comparison_eegnet(loss_weights=class_weight_values)


def evaluate_model(model, model_name):
    probs = np.asarray(model.predict_proba(X_val), dtype=np.float64)
    if probs.shape != (len(y_val), len(CLASS_NAMES)) or not np.isfinite(probs).all():
        raise ValueError(f"Invalid validation probabilities from {model_name}: shape={probs.shape}")
    if not np.allclose(probs.sum(axis=1), 1.0, atol=1e-3):
        raise ValueError(f"Probability rows from {model_name} do not sum to 1.")
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    metrics = {
        "model": model_name,
        "validation_windows": int(len(y_val)),
        "accuracy": float(accuracy_score(y_val, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_val, pred)),
        "macro_f1": float(f1_score(y_val, pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "max_confidence": float(conf.max()),
        "accepted_rate_at_0.60": float((conf >= 0.60).mean()),
        "accepted_rate_at_0.70": float((conf >= 0.70).mean()),
    }
    return probs, pred, conf, metrics


paired_results = {}
paired_rows = []
for name, model, elapsed in [
    ("EEGNet unweighted (paired baseline)", paired_unweighted, paired_unweighted_seconds),
    ("EEGNet class-weighted CE", paired_weighted, paired_weighted_seconds),
]:
    probs, pred, conf, metrics = evaluate_model(model, name)
    metrics["training_seconds"] = float(elapsed)
    metrics["predicted_Rest"] = int((pred == 0).sum())
    metrics["predicted_Left_Hand"] = int((pred == 1).sum())
    metrics["predicted_Right_Hand"] = int((pred == 2).sum())
    paired_results[name] = {"model": model, "probabilities": probs, "prediction": pred, "confidence": conf}
    paired_rows.append(metrics)

paired_comparison = pd.DataFrame(paired_rows)
print("Paired comparison on the same R08 validation windows (not final-test results):")
display(paired_comparison.round(4))
paired_comparison.to_csv(OUTPUT_DIR / "class_weighted_model_comparison.csv", index=False)

for name, content in paired_results.items():
    print("\nConfusion matrix —", name)
    display(pd.DataFrame(
        confusion_matrix(y_val, content["prediction"], labels=[0, 1, 2]),
        index=[f"true_{label}" for label in CLASS_NAMES],
        columns=[f"pred_{label}" for label in CLASS_NAMES],
    ))
    print(classification_report(
        y_val, content["prediction"], labels=[0, 1, 2],
        target_names=CLASS_NAMES, zero_division=0,
    ))

# Evaluate the safety gate for both trained models. NaN wrong-movement rate means the model
# emitted zero movement commands, so the denominator is zero—not a measured 0% error rate.
paired_gate_rows = []
for name, content in paired_results.items():
    pred, conf = content["prediction"], content["confidence"]
    for threshold in (0.60, 0.70):
        accepted = conf >= threshold
        movement = accepted & np.isin(pred, [1, 2])
        true_rest = y_val == 0
        wrong_movement = movement & (pred != y_val)
        paired_gate_rows.append({
            "model": name,
            "threshold": threshold,
            "accepted_output_rate": float(accepted.mean()),
            "no_command_rate": float((~accepted).mean()),
            "movement_commands": int(movement.sum()),
            "false_movement_on_true_rest_rate": float((movement & true_rest).sum() / true_rest.sum()) if true_rest.any() else np.nan,
            "wrong_movement_command_rate": float(wrong_movement.sum() / movement.sum()) if movement.sum() else np.nan,
        })
paired_gate_comparison = pd.DataFrame(paired_gate_rows)
print("Confidence-gate results for the paired models:")
display(paired_gate_comparison.round(4))
paired_gate_comparison.to_csv(OUTPUT_DIR / "class_weighted_gate_comparison.csv", index=False)

# Save both comparison candidates separately; keep the original first-run handoff unchanged.
ARTIFACT_DIR.mkdir(exist_ok=True)
for name, content in paired_results.items():
    slug = "unweighted" if "unweighted" in name else "class_weighted"
    weights_file = ARTIFACT_DIR / f"neuromotion_eegnet_{slug}_comparison_state_dict.pt"
    state_dict_cpu = {key: tensor.detach().cpu() for key, tensor in content["model"].module_.state_dict().items()}
    torch.save(state_dict_cpu, weights_file)
    print("Saved comparison weights:", weights_file.resolve())

weighted_config = {
    "experiment": "paired unweighted versus class-weighted cross-entropy",
    "single_intended_change": "CrossEntropyLoss class weights inversely proportional to training class counts",
    "class_names": CLASS_NAMES,
    "class_weights_by_class": {CLASS_NAMES[i]: float(class_weight_values[i]) for i in range(len(CLASS_NAMES))},
    "training_run": list(TRAIN_RUNS),
    "validation_run": list(VALIDATION_RUNS),
    "reserved_test_run_not_loaded": list(RESERVED_TEST_RUNS),
    "seed": SEED,
    "configuration": INITIAL_CONFIG,
    "note": "Development comparison only. Do not use R12 until the model and gate are frozen.",
}
with open(ARTIFACT_DIR / "neuromotion_class_weighted_comparison_config.json", "w", encoding="utf-8") as f:
    json.dump(weighted_config, f, indent=2)

# Save paired per-window predictions for auditability and error analysis.
paired_prediction_frame = meta_val.copy().reset_index(drop=True)
paired_prediction_frame["true_class"] = [CLASS_NAMES[int(v)] for v in y_val]
for name, content in paired_results.items():
    slug = "unweighted" if "unweighted" in name else "class_weighted"
    paired_prediction_frame[f"{slug}_prediction"] = [CLASS_NAMES[int(v)] for v in content["prediction"]]
    paired_prediction_frame[f"{slug}_confidence"] = content["confidence"]
    paired_prediction_frame[f"{slug}_correct"] = content["prediction"] == y_val
paired_prediction_frame.to_csv(OUTPUT_DIR / "class_weighted_validation_predictions.csv", index=False)

print("Saved comparison metrics and predictions to:", OUTPUT_DIR.resolve())
print("Interpretation: class weighting only helps if the measured validation class metrics/predictions improve without an unacceptable command-safety trade-off. Do not claim an improvement unless the printed tables support it.")


Training counts by class: {'Rest': 15, 'Left Hand': 8, 'Right Hand': 7}
Class weights: {'Rest': 0.6667, 'Left Hand': 1.25, 'Right Hand': 1.4286}
Paired comparison on the same R08 validation windows (not final-test results):


,model,validation_windows,accuracy,balanced_accuracy,macro_f1,max_confidence,accepted_rate_at_0.60,accepted_rate_at_0.70,training_seconds,predicted_Rest,predicted_Left_Hand,predicted_Right_Hand
0,EEGNet unweighted (paired baseline),30,0.5000,0.3333,0.2222,0.3762,0.0,0.0,0.1579,30,0,0
1,EEGNet class-weighted CE,30,0.4667,0.3948,0.3909,0.3535,0.0,0.0,0.1624,18,7,5



Confusion matrix — EEGNet unweighted (paired baseline)


,pred_Rest,pred_Left Hand,pred_Right Hand
true_Rest,15,0,0
true_Left Hand,8,0,0
true_Right Hand,7,0,0


              precision    recall  f1-score   support

        Rest       0.50      1.00      0.67        15
   Left Hand       0.00      0.00      0.00         8
  Right Hand       0.00      0.00      0.00         7

    accuracy                           0.50        30
   macro avg       0.17      0.33      0.22        30
weighted avg       0.25      0.50      0.33        30


Confusion matrix — EEGNet class-weighted CE


,pred_Rest,pred_Left Hand,pred_Right Hand
true_Rest,10,2,3
true_Left Hand,4,3,1
true_Right Hand,4,2,1


              precision    recall  f1-score   support

        Rest       0.56      0.67      0.61        15
   Left Hand       0.43      0.38      0.40         8
  Right Hand       0.20      0.14      0.17         7

    accuracy                           0.47        30
   macro avg       0.39      0.39      0.39        30
weighted avg       0.44      0.47      0.45        30

Confidence-gate results for the paired models:


,model,threshold,accepted_output_rate,no_command_rate,movement_commands,false_movement_on_true_rest_rate,wrong_movement_command_rate
0,EEGNet unweighted (paired baseline),0.6,0.0,1.0,0,0.0,NaN
1,EEGNet unweighted (paired baseline),0.7,0.0,1.0,0,0.0,NaN
2,EEGNet class-weighted CE,0.6,0.0,1.0,0,0.0,NaN
3,EEGNet class-weighted CE,0.7,0.0,1.0,0,0.0,NaN


Saved comparison weights: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_unweighted_comparison_state_dict.pt
Saved comparison weights: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\artifacts\neuromotion_eegnet_class_weighted_comparison_state_dict.pt
Saved comparison metrics and predictions to: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\outputs
Interpretation: class weighting only helps if the measured validation class metrics/predictions improve without an unacceptable command-safety trade-off. Do not claim an improvement unless the printed tables support it.


### How to report the follow-up

Use `class_weighted_model_comparison.csv` and the confusion matrices. Compare macro-F1 and balanced accuracy first, then inspect each class's recall and the number of windows predicted as each class. Use the gate table to report both the false-movement rate and usable/accepted-output rate. If class weighting still predicts Rest for every window, or if it creates movement commands with unacceptable validation errors, document that negative result and move to a CSP + LDA baseline / more training runs rather than tuning only the confidence gate.

The two comparison artifacts are saved with distinct names and do not overwrite the initial handoff. The candidate ultimately selected for the replay system should be chosen only after development comparisons are complete, then evaluated on the untouched R12 test run once.


## Limitations and next actions

1. The initial measured run is a failed baseline: it predicted Rest for every R08 validation window (accuracy 0.50 equals the majority baseline; balanced accuracy 0.333; macro-F1 0.222). The paired class-weighted follow-up must be run before deciding whether the training change helps.
2. This is a small, within-subject development experiment: only 30 windows are in each run. The resulting metric variance will be high and should not be generalized to other people.
3. `R12` is reserved for the future final test and must remain unused until the model/configuration and confidence threshold are frozen.
4. The project must compare EEGNet against a simpler CSP + LDA baseline and/or ShallowFBCSPNet on the same development split before deciding that the neural approach is preferable.
5. Confidence is not calibrated probability of correctness. The confidence gate is a heuristic until calibrated on appropriate validation data. High-confidence wrong predictions remain possible; the initial 0.60/0.70 gate comparison is uninformative because every confidence was below 0.60.
6. The corrected latency cell must be rerun because it now warms up the model and synchronizes CUDA around timed calls. The measurement still excludes EDF reading, streaming filters, UI rendering, and replay orchestration.
7. Do not interpret model predictions as real prosthetic control. The current deliverable supports offline recorded-EEG replay only.

## References

- Braindecode documentation, *Simple training on MNE epochs*: https://braindecode.org/stable/auto_examples/model_building/plot_basic_training_epochs.html
- Braindecode API, `EEGNet`: https://braindecode.org/stable/generated/braindecode.models.EEGNet.html
- Braindecode API, `EEGClassifier`: https://braindecode.org/stable/generated/braindecode.classifier.EEGClassifier.html
- PhysioNet EEG Motor Movement/Imagery Dataset v1.0.0: https://physionet.org/content/eegmmidb/1.0.0/ (DOI: 10.13026/C28G6P)
- Schalk, G. (2009). *EEG Motor Movement/Imagery Dataset (version 1.0.0).* PhysioNet.
8. A separate all-subject experiment is added at the end of this notebook. Its result must be reported independently from the original S001 run-level experiment. It uses subject-grouped five-fold cross-validation; it does not establish live online performance or clinical suitability.

## Milestone 4 audit extension — R12 quality, run manifest, event manifest, and lineage

Run this after the original training/validation cells. It loads the reserved R12 recording only to inspect annotations, labels, signal quality, and provenance. It does **not** use R12 for fitting, threshold selection, model comparison, or prediction. Keep R12 reserved until the model and threshold are frozen. This cell writes a run-level manifest, event/epoch manifest, group-overlap audit, raw-file SHA-256 values, and a lineage JSON record under `outputs/data_audit/`.


In [13]:
import hashlib
import importlib.metadata
from datetime import datetime, timezone

DATA_AUDIT_DIR = OUTPUT_DIR / "data_audit"
DATA_AUDIT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Load the reserved test run for data-quality inspection only. Do not pass it to net.fit,
# threshold selection, or model comparison.
test_run_data = {run: load_run(SUBJECT, run) for run in RESERVED_TEST_RUNS}

split_records = []
event_frames = []
all_run_data = {}
for run in TRAIN_RUNS:
    all_run_data[run] = (run_data[run], "TRAIN")
for run in VALIDATION_RUNS:
    all_run_data[run] = (run_data[run], "VALIDATION")
for run in RESERVED_TEST_RUNS:
    all_run_data[run] = (test_run_data[run], "TEST_RESERVED")

for run, (loaded, split_name) in all_run_data.items():
    X_run, y_run, meta_run, channel_names_run, sfreq_run, report_run = loaded
    counts = pd.Series([CLASS_NAMES[int(v)] for v in y_run]).value_counts().reindex(CLASS_NAMES, fill_value=0)
    path = find_recording(SUBJECT, run)
    split_records.append({
        **report_run,
        "split": split_name,
        "rest_count": int(counts["Rest"]),
        "left_hand_count": int(counts["Left Hand"]),
        "right_hand_count": int(counts["Right Hand"]),
        "file_size_bytes": int(path.stat().st_size),
        "file_sha256": sha256_file(path),
    })
    frame = meta_run.copy().reset_index(drop=True)
    frame.insert(0, "split", split_name)
    frame.insert(0, "event_id", [
        f"{SUBJECT}-{run}-{float(onset):.3f}s-{annotation}"
        for onset, annotation in zip(frame["event_onset_sec"], frame["annotation"])
    ])
    event_frames.append(frame)
    if channel_names_run != reference_channels:
        raise ValueError(f"Channel names/order differ in {run}; do not combine this run until harmonized.")

run_manifest_df = pd.DataFrame(split_records)
event_manifest_df = pd.concat(event_frames, ignore_index=True)

train_group_ids = {f"{SUBJECT}{run}" for run in TRAIN_RUNS}
validation_group_ids = {f"{SUBJECT}{run}" for run in VALIDATION_RUNS}
test_group_ids = {f"{SUBJECT}{run}" for run in RESERVED_TEST_RUNS}

train_val_overlap = sorted(train_group_ids & validation_group_ids)
train_test_overlap = sorted(train_group_ids & test_group_ids)
validation_test_overlap = sorted(validation_group_ids & test_group_ids)
all_event_ids_unique = bool(event_manifest_df["event_id"].is_unique)

# The split is defined at the complete-run group level before model windows are created.
# The event manifest makes each retained annotation-derived epoch traceable to its parent run.
split_audit = {
    "train_run_ids": sorted(train_group_ids),
    "validation_run_ids": sorted(validation_group_ids),
    "test_run_ids_reserved": sorted(test_group_ids),
    "train_validation_overlap": train_val_overlap,
    "train_test_overlap": train_test_overlap,
    "validation_test_overlap": validation_test_overlap,
    "run_group_zero_overlap": not (train_val_overlap or train_test_overlap or validation_test_overlap),
    "all_event_ids_unique": all_event_ids_unique,
    "R12_used_for_model_selection_or_threshold_tuning": False,
    "note": "R12 was loaded only for data-quality/label/lineage inspection in this cell. Do not score it until model and threshold are frozen.",
}

run_manifest_df.to_csv(DATA_AUDIT_DIR / "run_level_split_manifest.csv", index=False)
event_manifest_df.to_csv(DATA_AUDIT_DIR / "event_epoch_manifest.csv", index=False)
with open(DATA_AUDIT_DIR / "split_audit.json", "w", encoding="utf-8") as f:
    json.dump(split_audit, f, indent=2)

environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "mne": mne.__version__,
    "scikit_learn": sklearn.__version__,
    "pytorch": torch.__version__,
    "braindecode": braindecode.__version__,
    "device": DEVICE,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
try:
    environment["torchaudio"] = importlib.metadata.version("torchaudio")
except importlib.metadata.PackageNotFoundError:
    environment["torchaudio"] = "not installed"

lineage = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "name": "PhysioNet EEG Motor Movement/Imagery Dataset (EEGMMIDB)",
        "version": "1.0.0",
        "url": "https://physionet.org/content/eegmmidb/1.0.0/",
        "doi": "10.13026/C28G6P",
        "license": "Open Data Commons Attribution License v1.0",
        "download_date": "unknown - not recorded at download time",
    },
    "subject": SUBJECT,
    "files": split_records,
    "annotation_to_class": {"T0": "Rest", "T1": "Left Hand imagery", "T2": "Right Hand imagery"},
    "split": split_audit,
    "processing": {
        "sampling_rate_hz": EXPECTED_SFREQ,
        "bandpass_hz": [LOW_CUT_HZ, HIGH_CUT_HZ],
        "epoch_tmin_seconds": WINDOW_TMIN,
        "epoch_tmax_seconds": WINDOW_TMAX,
        "reject_by_annotation": True,
        "amplitude_threshold_rejection": False,
        "normalization": "per-channel z-score; fit on training run(s) only",
        "seed": SEED,
    },
    "environment": environment,
    "model": INITIAL_CONFIG,
    "repository_url": "fill in the actual repository URL before submission",
    "git_commit": "fill in the exact commit that produced the reported experiment",
}
with open(DATA_AUDIT_DIR / "lineage_record.json", "w", encoding="utf-8") as f:
    json.dump(lineage, f, indent=2)

print("Run-level manifest:")
display(run_manifest_df[["subject", "run", "split", "requested_events_T0_T1_T2", "retained_epochs", "dropped_epochs", "rest_count", "left_hand_count", "right_hand_count", "finite_values", "min_uV", "max_uV"]])
print("Run-group overlap audit:", split_audit)
print("Event-to-epoch examples:")
display(event_manifest_df.groupby("true_class", group_keys=False).head(2))
print("Saved audit files to:", DATA_AUDIT_DIR.resolve())


Run-level manifest:


,subject,run,split,requested_events_T0_T1_T2,retained_epochs,dropped_epochs,rest_count,left_hand_count,right_hand_count,finite_values,min_uV,max_uV
0,S001,R04,TRAIN,30,30,0,15,8,7,True,-315.934937,305.064911
1,S001,R08,VALIDATION,30,30,0,15,8,7,True,-251.535553,296.216949
2,S001,R12,TEST_RESERVED,30,30,0,15,7,8,True,-183.445923,265.879578


Run-group overlap audit: {'train_run_ids': ['S001R04'], 'validation_run_ids': ['S001R08'], 'test_run_ids_reserved': ['S001R12'], 'train_validation_overlap': [], 'train_test_overlap': [], 'validation_test_overlap': [], 'run_group_zero_overlap': True, 'all_event_ids_unique': True, 'R12_used_for_model_selection_or_threshold_tuning': False, 'note': 'R12 was loaded only for data-quality/label/lineage inspection in this cell. Do not score it until model and threshold are frozen.'}
Event-to-epoch examples:


,event_id,split,subject,run,event_onset_sec,annotation,label_id,true_class
0,S001-R04-0.000s-T0,TRAIN,S001,R04,0.0,T0,0,Rest
1,S001-R04-4.200s-T2,TRAIN,S001,R04,4.2,T2,2,Right Hand
2,S001-R04-8.300s-T0,TRAIN,S001,R04,8.3,T0,0,Rest
3,S001-R04-12.500s-T1,TRAIN,S001,R04,12.5,T1,1,Left Hand
5,S001-R04-20.800s-T1,TRAIN,S001,R04,20.8,T1,1,Left Hand
7,S001-R04-29.100s-T2,TRAIN,S001,R04,29.1,T2,2,Right Hand


Saved audit files to: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\outputs\data_audit


### Submission status note
The baseline outputs above are the previously executed measurements. The synchronized latency cell and the new R12/manifest/lineage audit cell must be rerun locally; their results are intentionally not fabricated here. The class-weighted comparison must also be executed before it can be included as a measured model result.


## Additional experiment — train and test across all available subjects

The original S001 experiment answers a **within-subject** question. This additional experiment answers a different and more demanding question: **does an EEGNet trained on some participants generalize to participants it has not seen?**

The PhysioNet EEGMMIDB v1.0.0 dataset contains recordings from 109 volunteers. This experiment attempts to inspect all subject IDs `S001` through `S109` and use the unilateral **imagined** fist runs `R04`, `R08`, and `R12`. For these runs the task mapping is `T0 = Rest`, `T1 = Left Hand imagery`, and `T2 = Right Hand imagery`. Runs with missing files, invalid sampling rates, invalid labels, non-finite values, or incompatible channels are logged. A participant is eligible only if at least one usable run remains and the participant's combined epochs contain all three classes.

### Important: split by participant, not by EEG window

A random split of all windows would leak participant-specific EEG patterns into both training and test sets. Instead, `StratifiedGroupKFold` assigns each participant to exactly one held-out test fold. With five folds, each eligible participant is tested once and contributes to the training data in the other four folds. All runs and windows from one participant stay together within a fold. This makes the out-of-fold aggregate a subject-independent evaluation over the eligible participants.

This is **not** one model trained on every participant and then tested on those same participants. That would be an invalid evaluation. In each fold, the model trains on approximately 80% of the eligible participants and tests on the remaining approximately 20%; fold-level training/test subject IDs and overlap checks are saved.

### Preprocessing and evaluation rules

- Filter each recording at 4–38 Hz and epoch from 0.5–3.5 seconds after each selected event, matching the original experiment.
- Align channel order to the original S001 reference channel order. If the channel set differs, log and exclude that run rather than silently changing the model input.
- Fit per-channel normalization using only the training participants in the current fold. Apply those same statistics to that fold's held-out participants.
- Keep the EEGNet architecture, Adam optimizer, learning rate 0.001, and 10 epochs. Use batch size 32 for this larger multi-subject experiment and record that configuration separately from the original batch-size-8 run.
- Use the 0.60 and 0.70 gates as fixed diagnostic thresholds only. Do not select a threshold based on an outer test fold.
- Report accuracy, balanced accuracy, macro-F1, confusion matrix, majority-class baseline, per-subject metrics, and gate utility/safety measures. In particular, report accepted output and correct movement-command rates so high accuracy from REST predictions cannot disguise a controller that produces no useful commands.

**Scope note:** this experiment measures cross-subject generalization within this dataset and task. It does not prove performance on a new recording device, new protocol, live EEG, or clinical/prosthetic use.


In [14]:
# All-subject data ingestion and audit. This cell attempts S001-S109, not just subjects with known-good files.
import gc
import hashlib
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score

ALL_SUBJECT_IDS = [f"S{i:03d}" for i in range(1, 110)]
ALL_SUBJECT_RUNS = ("R04", "R08", "R12")
ALL_SUBJECT_HASH_FILES = True  # Set False only for a quick development run; True creates stronger lineage evidence.
ALL_SUBJECT_OUTPUT_DIR = OUTPUT_DIR / "all_subject_cross_validation"
ALL_SUBJECT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file_for_cv(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


all_subject_X_blocks = []
all_subject_y_blocks = []
all_subject_meta_blocks = []
all_subject_run_records = []
all_subject_summary_records = []
all_subject_file_records = []

for subject in ALL_SUBJECT_IDS:
    subject_X_runs = []
    subject_y_runs = []
    subject_meta_runs = []
    subject_valid_runs = 0

    for run in ALL_SUBJECT_RUNS:
        run_record = {"subject": subject, "run": run, "status": "NOT_PROCESSED"}
        try:
            recording_path = find_recording(subject, run)
            X_run, y_run, meta_run, channel_names, sfreq, report = load_run(subject, run)

            # Keep a consistent model input across participants. Reorder if the channel set
            # is identical; exclude a run if any channel is missing or extra.
            if set(channel_names) != set(reference_channels) or len(channel_names) != len(reference_channels):
                raise ValueError(
                    f"Channel set differs from reference: got {len(channel_names)} channels; "
                    f"reference has {len(reference_channels)}."
                )
            channel_order = [channel_names.index(name) for name in reference_channels]
            X_run = X_run[:, channel_order, :]

            if X_run.ndim != 3 or X_run.shape[1] != len(reference_channels):
                raise ValueError(f"Unexpected epoch tensor shape: {X_run.shape}")
            if X_run.shape[2] != int(round((WINDOW_TMAX - WINDOW_TMIN) * EXPECTED_SFREQ)) + 1:
                raise ValueError(f"Unexpected samples per epoch: {X_run.shape[2]}")
            if not np.isfinite(X_run).all():
                raise ValueError("Non-finite EEG values after epoch extraction.")
            if len(y_run) == 0 or not set(np.unique(y_run)).issubset({0, 1, 2}):
                raise ValueError(f"No usable epochs or unexpected labels: {np.unique(y_run)}")

            counts = np.bincount(y_run, minlength=len(CLASS_NAMES))
            peak_abs_uv = float(np.max(np.abs(X_run))) if X_run.size else np.nan
            samples_over_500_uv = int(np.count_nonzero(np.abs(X_run) > 500.0))
            file_hash = sha256_file_for_cv(recording_path) if ALL_SUBJECT_HASH_FILES else "not computed"

            run_record.update({
                **report,
                "status": "LOADED",
                "rest_count": int(counts[0]),
                "left_hand_count": int(counts[1]),
                "right_hand_count": int(counts[2]),
                "peak_absolute_uV": peak_abs_uv,
                "samples_over_500uV_diagnostic": samples_over_500_uv,
                "file_size_bytes": int(recording_path.stat().st_size),
                "file_sha256": file_hash,
                "channel_order_aligned_to_reference": True,
                "error": "",
            })

            subject_X_runs.append(X_run)
            subject_y_runs.append(y_run)
            subject_meta_runs.append(meta_run.copy())
            subject_valid_runs += 1
            all_subject_file_records.append({
                "subject": subject,
                "run": run,
                "path": str(recording_path),
                "sha256": file_hash,
                "file_size_bytes": int(recording_path.stat().st_size),
            })

        except FileNotFoundError as exc:
            run_record.update({"status": "MISSING_FILE", "error": str(exc)})
        except Exception as exc:
            # Preserve the failure in the audit rather than silently treating it as valid data.
            run_record.update({"status": "EXCLUDED_ERROR", "error": f"{type(exc).__name__}: {exc}"})

        all_subject_run_records.append(run_record)

    if subject_valid_runs == 0:
        all_subject_summary_records.append({
            "subject": subject, "included_in_cv": False, "valid_runs": 0,
            "retained_epochs": 0, "rest_count": 0, "left_hand_count": 0,
            "right_hand_count": 0, "reason": "No usable R04/R08/R12 recording found."
        })
        continue

    X_subject = np.concatenate(subject_X_runs, axis=0).astype(np.float32, copy=False)
    y_subject = np.concatenate(subject_y_runs, axis=0).astype(np.int64, copy=False)
    meta_subject = pd.concat(subject_meta_runs, ignore_index=True)
    subject_classes = set(np.unique(y_subject).tolist())
    subject_counts = np.bincount(y_subject, minlength=len(CLASS_NAMES))

    if subject_classes != {0, 1, 2}:
        all_subject_summary_records.append({
            "subject": subject, "included_in_cv": False, "valid_runs": subject_valid_runs,
            "retained_epochs": int(len(y_subject)), "rest_count": int(subject_counts[0]),
            "left_hand_count": int(subject_counts[1]), "right_hand_count": int(subject_counts[2]),
            "reason": "Combined usable runs do not contain all three classes."
        })
        continue

    meta_subject.insert(0, "event_id", [
        f"{subject}-{run}-{float(onset):.3f}s-{annotation}"
        for run, onset, annotation in zip(
            meta_subject["run"], meta_subject["event_onset_sec"], meta_subject["annotation"]
        )
    ])
    all_subject_X_blocks.append(X_subject)
    all_subject_y_blocks.append(y_subject)
    all_subject_meta_blocks.append(meta_subject)
    all_subject_summary_records.append({
        "subject": subject, "included_in_cv": True, "valid_runs": subject_valid_runs,
        "retained_epochs": int(len(y_subject)), "rest_count": int(subject_counts[0]),
        "left_hand_count": int(subject_counts[1]), "right_hand_count": int(subject_counts[2]),
        "reason": "Included; at least one valid run and all three classes present."
    })

if len(all_subject_X_blocks) < 5:
    raise RuntimeError(
        f"Only {len(all_subject_X_blocks)} eligible subjects were found; five-fold subject CV needs at least five. "
        "Review the all_subject_run_quality.csv audit before continuing."
    )

X_all_subjects = np.concatenate(all_subject_X_blocks, axis=0).astype(np.float32, copy=False)
y_all_subjects = np.concatenate(all_subject_y_blocks, axis=0).astype(np.int64, copy=False)
meta_all_subjects = pd.concat(all_subject_meta_blocks, ignore_index=True)
groups_all_subjects = meta_all_subjects["subject"].astype(str).to_numpy()
# The concatenated arrays now own the data. Release the per-subject EEG arrays before CV
# to avoid keeping a second full copy in RAM.
del all_subject_X_blocks, all_subject_y_blocks


all_subject_run_quality_df = pd.DataFrame(all_subject_run_records)
all_subject_summary_df = pd.DataFrame(all_subject_summary_records)
all_subject_file_manifest_df = pd.DataFrame(all_subject_file_records)
all_subject_run_quality_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "all_subject_run_quality.csv", index=False)
all_subject_summary_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "all_subject_summary.csv", index=False)
all_subject_file_manifest_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "all_subject_file_manifest.csv", index=False)
meta_all_subjects.to_csv(ALL_SUBJECT_OUTPUT_DIR / "all_subject_event_epoch_manifest.csv", index=False)

print(f"Subjects discovered/attempted: {len(ALL_SUBJECT_IDS)}")
print(f"Eligible subjects included in CV: {all_subject_summary_df['included_in_cv'].sum()}")
print(f"Subjects excluded: {(~all_subject_summary_df['included_in_cv']).sum()}")
print(f"Runs loaded: {(all_subject_run_quality_df['status'] == 'LOADED').sum()}")
print(f"Runs missing: {(all_subject_run_quality_df['status'] == 'MISSING_FILE').sum()}")
print(f"Runs rejected due to data/format errors: {(all_subject_run_quality_df['status'] == 'EXCLUDED_ERROR').sum()}")
print("Combined EEG tensor (epochs, channels, time):", X_all_subjects.shape)
print("Combined class counts:")
display(pd.Series(y_all_subjects).map(dict(enumerate(CLASS_NAMES))).value_counts().reindex(CLASS_NAMES, fill_value=0).rename("epochs").to_frame())
print("Per-subject inclusion summary (first 15 rows):")
display(all_subject_summary_df.head(15))
print("Run-quality audit (first 15 rows):")
display(all_subject_run_quality_df.head(15))

assert np.isfinite(X_all_subjects).all(), "Combined dataset contains non-finite values."
assert set(np.unique(y_all_subjects)) == {0, 1, 2}, "Combined dataset is missing one or more classes."
assert meta_all_subjects["event_id"].is_unique, "Duplicate subject-run-event IDs were found."


Subjects discovered/attempted: 109
Eligible subjects included in CV: 106
Subjects excluded: 3
Runs loaded: 318
Runs missing: 0
Runs rejected due to data/format errors: 9
Combined EEG tensor (epochs, channels, time): (9535, 64, 481)
Combined class counts:


,epochs
Rest,4768
Left Hand,2405
Right Hand,2362


Per-subject inclusion summary (first 15 rows):


,subject,included_in_cv,valid_runs,retained_epochs,rest_count,left_hand_count,right_hand_count,reason
0,S001,True,3,90,45,23,22,Included; at least one valid run and all three...
1,S002,True,3,90,45,23,22,Included; at least one valid run and all three...
2,S003,True,3,90,45,23,22,Included; at least one valid run and all three...
3,S004,True,3,90,45,23,22,Included; at least one valid run and all three...
4,S005,True,3,90,45,21,24,Included; at least one valid run and all three...
5,S006,True,3,90,45,24,21,Included; at least one valid run and all three...
6,S007,True,3,90,45,23,22,Included; at least one valid run and all three...
7,S008,True,3,90,45,22,23,Included; at least one valid run and all three...
8,S009,True,3,90,45,24,21,Included; at least one valid run and all three...
9,S010,True,3,90,45,24,21,Included; at least one valid run and all three...


Run-quality audit (first 15 rows):


,subject,run,status,file,sampling_rate_hz,channels,requested_events_T0_T1_T2,retained_epochs,dropped_epochs,samples_per_epoch,...,max_uV,rest_count,left_hand_count,right_hand_count,peak_absolute_uV,samples_over_500uV_diagnostic,file_size_bytes,file_sha256,channel_order_aligned_to_reference,error
0,S001,R04,LOADED,data\S001\S001R04.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,305.064911,15.0,8.0,7.0,315.934937,0.0,2596896.0,3d161f88e1c00632585287d2ce584c2bc0f08862438eb2...,True,
1,S001,R08,LOADED,data\S001\S001R08.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,296.216949,15.0,8.0,7.0,296.216949,0.0,2596896.0,358fb5189220725141968ae285fbe9e3f36210b834ffba...,True,
2,S001,R12,LOADED,data\S001\S001R12.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,265.879578,15.0,7.0,8.0,265.879578,0.0,2596896.0,2b281c9b687b4c4176e83251d74743721f2d6ebd76656a...,True,
3,S002,R04,LOADED,data\S002\S002R04.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,162.638733,15.0,7.0,8.0,162.638733,0.0,2555616.0,fcd37831378c411d50c223d97ffbc00949be2271d093f1...,True,
4,S002,R08,LOADED,data\S002\S002R08.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,103.780479,15.0,8.0,7.0,103.780479,0.0,2555616.0,02d64941f6bcd1635bc7dd187a9553331b73933e9771f4...,True,
5,S002,R12,LOADED,data\S002\S002R12.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,143.675262,15.0,8.0,7.0,181.549454,0.0,2555616.0,a3d166b23375942a5ccf352924f7766f0ca9cfd1bac795...,True,
6,S003,R04,LOADED,data\S003\S003R04.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,387.645447,15.0,8.0,7.0,387.645447,0.0,2596896.0,7d0732eea963488a53153835524e55c2b68220b0a0c7c5...,True,
7,S003,R08,LOADED,data\S003\S003R08.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,434.632416,15.0,7.0,8.0,434.632416,0.0,2596896.0,0f50978bb972e693b8c758a9223a2d9fa35c7f11722639...,True,
8,S003,R12,LOADED,data\S003\S003R12.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,453.009430,15.0,8.0,7.0,459.908661,0.0,2596896.0,bbf7137bfa7905724741e95359fa090439d4422c07bdba...,True,
9,S004,R04,LOADED,data\S004\S004R04.edf,160.0,64.0,30.0,30.0,0.0,481.0,...,270.566772,15.0,8.0,7.0,361.977966,0.0,2555616.0,c9ddf0294961f0b877192a2479802b9c6c88403682a17b...,True,


In [15]:
## Run subject-independent five-fold cross-validation

# Each participant is assigned to one and only one outer test fold. The group is the participant,
# not the recording or window, so all three runs and all overlapping/correlated epochs remain together.
N_SUBJECT_FOLDS = 5
ALL_SUBJECT_EPOCHS = 10
ALL_SUBJECT_BATCH_SIZE = 32
ALL_SUBJECT_LEARNING_RATE = 0.001
ALL_SUBJECT_THRESHOLDS = (0.60, 0.70)  # fixed diagnostics; no test-fold tuning

cv = StratifiedGroupKFold(n_splits=N_SUBJECT_FOLDS, shuffle=True, random_state=SEED)

oof_probabilities = np.full((len(y_all_subjects), len(CLASS_NAMES)), np.nan, dtype=np.float32)
oof_predictions = np.full(len(y_all_subjects), -1, dtype=np.int64)
oof_majority_predictions = np.full(len(y_all_subjects), -1, dtype=np.int64)
fold_metric_records = []
fold_subject_records = []
fold_assignment_records = []
fold_confusion_records = []

for fold_number, (train_indices, test_indices) in enumerate(
    cv.split(X_all_subjects, y_all_subjects, groups=groups_all_subjects), start=1
):
    train_subjects = sorted(set(groups_all_subjects[train_indices]))
    test_subjects = sorted(set(groups_all_subjects[test_indices]))
    overlap = sorted(set(train_subjects) & set(test_subjects))
    if overlap:
        raise RuntimeError(f"Subject leakage in fold {fold_number}: {overlap}")

    # Fit normalization statistics on the training subjects in this fold only.
    training_mean = X_all_subjects[train_indices].mean(axis=(0, 2), keepdims=True)
    training_std = X_all_subjects[train_indices].std(axis=(0, 2), keepdims=True)
    training_std = np.where(training_std < 1e-6, 1.0, training_std)

    X_train_fold = X_all_subjects[train_indices].astype(np.float32, copy=True)
    X_test_fold = X_all_subjects[test_indices].astype(np.float32, copy=True)
    X_train_fold -= training_mean
    X_train_fold /= training_std
    X_test_fold -= training_mean
    X_test_fold /= training_std
    y_train_fold = y_all_subjects[train_indices]
    y_test_fold = y_all_subjects[test_indices]

    if set(np.unique(y_train_fold)) != {0, 1, 2}:
        raise RuntimeError(f"Training fold {fold_number} is missing a class.")
    if set(np.unique(y_test_fold)) != {0, 1, 2}:
        raise RuntimeError(f"Test fold {fold_number} is missing a class.")

    # Re-seed each fold so the run is deterministic and independently reproducible.
    fold_seed = SEED + fold_number
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    fold_model = EEGClassifier(
        "EEGNet",
        module__n_chans=int(X_train_fold.shape[1]),
        module__n_outputs=len(CLASS_NAMES),
        module__n_times=int(X_train_fold.shape[2]),
        module__sfreq=float(EXPECTED_SFREQ),
        criterion=torch.nn.CrossEntropyLoss,
        optimizer=torch.optim.Adam,
        lr=ALL_SUBJECT_LEARNING_RATE,
        max_epochs=ALL_SUBJECT_EPOCHS,
        batch_size=ALL_SUBJECT_BATCH_SIZE,
        train_split=None,
        iterator_train__shuffle=True,
        iterator_train__drop_last=False,
        device=DEVICE,
        verbose=0,
    )

    fold_start = time.perf_counter()
    fold_model.fit(X_train_fold, y_train_fold)
    fold_training_seconds = time.perf_counter() - fold_start

    fold_probs = np.asarray(fold_model.predict_proba(X_test_fold), dtype=np.float32)
    if fold_probs.shape != (len(test_indices), len(CLASS_NAMES)):
        raise RuntimeError(f"Unexpected test probability shape in fold {fold_number}: {fold_probs.shape}")
    if not np.isfinite(fold_probs).all() or not np.allclose(fold_probs.sum(axis=1), 1.0, atol=1e-3):
        raise RuntimeError(f"Invalid test probabilities in fold {fold_number}.")

    fold_pred = fold_probs.argmax(axis=1)
    oof_probabilities[test_indices] = fold_probs
    oof_predictions[test_indices] = fold_pred

    majority_label = int(np.bincount(y_train_fold, minlength=len(CLASS_NAMES)).argmax())
    majority_pred = np.full_like(y_test_fold, majority_label)
    oof_majority_predictions[test_indices] = majority_pred

    fold_metric_records.append({
        "fold": fold_number,
        "train_subjects": len(train_subjects),
        "test_subjects": len(test_subjects),
        "train_epochs": len(train_indices),
        "test_epochs": len(test_indices),
        "training_seconds": fold_training_seconds,
        "accuracy": float(accuracy_score(y_test_fold, fold_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test_fold, fold_pred)),
        "macro_f1": float(f1_score(y_test_fold, fold_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "majority_baseline_accuracy": float(accuracy_score(y_test_fold, majority_pred)),
        "train_test_subject_overlap": len(overlap),
    })
    for subject in train_subjects:
        fold_assignment_records.append({"fold": fold_number, "subject": subject, "role": "TRAIN"})
    for subject in test_subjects:
        fold_subject_records.append({"subject": subject, "held_out_test_fold": fold_number})
        fold_assignment_records.append({"fold": fold_number, "subject": subject, "role": "TEST"})

    fold_cm = confusion_matrix(y_test_fold, fold_pred, labels=[0, 1, 2])
    for true_idx, true_name in enumerate(CLASS_NAMES):
        for pred_idx, pred_name in enumerate(CLASS_NAMES):
            fold_confusion_records.append({
                "fold": fold_number,
                "true_class": true_name,
                "predicted_class": pred_name,
                "count": int(fold_cm[true_idx, pred_idx]),
            })

    print(
        f"Fold {fold_number}/{N_SUBJECT_FOLDS}: "
        f"train={len(train_subjects)} subjects / {len(train_indices)} epochs; "
        f"test={len(test_subjects)} subjects / {len(test_indices)} epochs; "
        f"accuracy={accuracy_score(y_test_fold, fold_pred):.3f}, "
        f"macro-F1={f1_score(y_test_fold, fold_pred, labels=[0, 1, 2], average='macro', zero_division=0):.3f}, "
        f"training={fold_training_seconds:.1f}s"
    )

    del fold_model, X_train_fold, X_test_fold, y_train_fold, y_test_fold, fold_probs, fold_pred
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if np.any(~np.isfinite(oof_probabilities)) or np.any(oof_predictions < 0) or np.any(oof_majority_predictions < 0):
    raise RuntimeError("Some subjects/windows did not receive exactly one out-of-fold prediction.")

fold_metrics_df = pd.DataFrame(fold_metric_records)
fold_subject_manifest_df = pd.DataFrame(fold_subject_records).sort_values("subject").reset_index(drop=True)
fold_confusion_long_df = pd.DataFrame(fold_confusion_records)
fold_assignment_manifest_df = pd.DataFrame(fold_assignment_records)
if fold_subject_manifest_df["subject"].duplicated().any():
    raise RuntimeError("A subject was assigned to the outer test set more than once.")
if set(fold_subject_manifest_df["subject"]) != set(np.unique(groups_all_subjects)):
    raise RuntimeError("Not every eligible subject appears exactly once in the test folds.")

# The fold test assignment plus the complementary train sets demonstrates the subject-level design.
fold_subject_manifest_df["included_in_training_folds"] = fold_subject_manifest_df["held_out_test_fold"].apply(
    lambda held_out: ",".join(str(fold) for fold in range(1, N_SUBJECT_FOLDS + 1) if fold != held_out)
)
fold_subject_manifest_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "subject_fold_manifest.csv", index=False)
fold_metrics_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "fold_metrics.csv", index=False)
fold_confusion_long_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "fold_confusion_matrices_long.csv", index=False)
fold_assignment_manifest_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "fold_subject_assignments.csv", index=False)

# Verify every subject is TEST once and TRAIN in every other fold.
expected_assignment_count = N_SUBJECT_FOLDS * len(np.unique(groups_all_subjects))
if len(fold_assignment_manifest_df) != expected_assignment_count:
    raise RuntimeError("Fold assignment manifest is incomplete.")
for subject, assignments in fold_assignment_manifest_df.groupby("subject"):
    if int((assignments["role"] == "TEST").sum()) != 1 or int((assignments["role"] == "TRAIN").sum()) != N_SUBJECT_FOLDS - 1:
        raise RuntimeError(f"Subject {subject} does not have exactly one test fold and four training folds.")
for fold, assignments in fold_assignment_manifest_df.groupby("fold"):
    train_ids = set(assignments.loc[assignments["role"] == "TRAIN", "subject"])
    test_ids = set(assignments.loc[assignments["role"] == "TEST", "subject"])
    if train_ids & test_ids:
        raise RuntimeError(f"Subject leakage in fold {fold}: {sorted(train_ids & test_ids)}")

# Out-of-fold aggregate: every eligible subject's epochs are evaluated only when that subject is held out.
all_true = y_all_subjects
all_pred = oof_predictions
all_conf = oof_probabilities.max(axis=1)
all_subject_overall_metrics = {
    "experiment": "All-subject EEGNet — 5-fold subject-independent CV",
    "eligible_subjects": int(len(np.unique(groups_all_subjects))),
    "excluded_subjects": int((~all_subject_summary_df["included_in_cv"]).sum()),
    "runs_permitted": list(ALL_SUBJECT_RUNS),
    "total_tested_epochs_out_of_fold": int(len(all_true)),
    "accuracy": float(accuracy_score(all_true, all_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(all_true, all_pred)),
    "macro_f1": float(f1_score(all_true, all_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
    "majority_baseline_oof_accuracy": float(accuracy_score(all_true, oof_majority_predictions)),
    "mean_fold_accuracy": float(fold_metrics_df["accuracy"].mean()),
    "std_fold_accuracy": float(fold_metrics_df["accuracy"].std(ddof=1)),
    "mean_fold_macro_f1": float(fold_metrics_df["macro_f1"].mean()),
    "std_fold_macro_f1": float(fold_metrics_df["macro_f1"].std(ddof=1)),
    "epochs_per_subject": int(ALL_SUBJECT_EPOCHS),
    "batch_size": int(ALL_SUBJECT_BATCH_SIZE),
    "learning_rate": float(ALL_SUBJECT_LEARNING_RATE),
    "seed_base": int(SEED),
    "device": DEVICE,
}

# Overall out-of-fold confusion matrix and class-level metrics.
all_subject_confusion = pd.DataFrame(
    confusion_matrix(all_true, all_pred, labels=[0, 1, 2]),
    index=[f"true_{name}" for name in CLASS_NAMES],
    columns=[f"pred_{name}" for name in CLASS_NAMES],
)
all_subject_confusion.to_csv(ALL_SUBJECT_OUTPUT_DIR / "oof_confusion_matrix.csv")

class_rows = []
for class_id, class_name in enumerate(CLASS_NAMES):
    true_mask = all_true == class_id
    predicted_mask = all_pred == class_id
    tp = int(np.sum(true_mask & predicted_mask))
    fp = int(np.sum(~true_mask & predicted_mask))
    fn = int(np.sum(true_mask & ~predicted_mask))
    class_rows.append({
        "class": class_name,
        "support": int(true_mask.sum()),
        "precision": tp / (tp + fp) if tp + fp else np.nan,
        "recall": tp / (tp + fn) if tp + fn else np.nan,
        "f1": (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else np.nan,
    })
class_metrics_all_subjects_df = pd.DataFrame(class_rows)
class_metrics_all_subjects_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "oof_class_metrics.csv", index=False)

# Per-subject metrics reveal whether aggregate performance hides substantial participant variation.
subject_metric_rows = []
meta_with_predictions = meta_all_subjects.copy()
meta_with_predictions["true_class"] = [CLASS_NAMES[int(v)] for v in all_true]
meta_with_predictions["predicted_class"] = [CLASS_NAMES[int(v)] for v in all_pred]
meta_with_predictions["confidence"] = all_conf
meta_with_predictions["correct"] = all_true == all_pred
meta_with_predictions["test_fold"] = meta_with_predictions["subject"].map(
    fold_subject_manifest_df.set_index("subject")["held_out_test_fold"]
)
for subject, subject_frame in meta_with_predictions.groupby("subject"):
    ys = subject_frame["label_id"].to_numpy(dtype=np.int64)
    ps = np.asarray([CLASS_NAMES.index(value) for value in subject_frame["predicted_class"]], dtype=np.int64)
    subject_metric_rows.append({
        "subject": subject,
        "test_fold": int(subject_frame["test_fold"].iloc[0]),
        "epochs": int(len(subject_frame)),
        "accuracy": float(accuracy_score(ys, ps)),
        "balanced_accuracy": float(balanced_accuracy_score(ys, ps)),
        "macro_f1": float(f1_score(ys, ps, labels=[0, 1, 2], average="macro", zero_division=0)),
        "mean_confidence": float(subject_frame["confidence"].mean()),
    })
subject_metrics_all_subjects_df = pd.DataFrame(subject_metric_rows).sort_values("subject")
subject_metrics_all_subjects_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "oof_subject_metrics.csv", index=False)
meta_with_predictions.to_csv(ALL_SUBJECT_OUTPUT_DIR / "oof_window_predictions.csv", index=False)

# Fixed-threshold gate diagnostics on the pooled out-of-fold predictions. These thresholds are not tuned here.
gate_rows = []
for threshold in ALL_SUBJECT_THRESHOLDS:
    accepted = all_conf >= threshold
    movement_commands = accepted & np.isin(all_pred, [1, 2])
    correct_movement_commands = movement_commands & (all_pred == all_true)
    wrong_movement_commands = movement_commands & (all_pred != all_true)
    true_rest = all_true == 0
    gate_rows.append({
        "threshold": threshold,
        "total_held_out_epochs": int(len(all_true)),
        "accepted_output_rate_including_rest": float(accepted.mean()),
        "no_command_rate": float((~accepted).mean()),
        "movement_command_rate": float(movement_commands.mean()),
        "correct_movement_command_rate_per_all_epochs": float(correct_movement_commands.mean()),
        "wrong_movement_commands": int(wrong_movement_commands.sum()),
        "wrong_movement_command_rate_among_movement_commands": (
            float(wrong_movement_commands.sum() / movement_commands.sum())
            if movement_commands.any() else np.nan
        ),
        "false_movement_on_true_rest_rate": (
            float((movement_commands & true_rest).sum() / true_rest.sum()) if true_rest.any() else np.nan
        ),
    })
gate_metrics_all_subjects_df = pd.DataFrame(gate_rows)
gate_metrics_all_subjects_df.to_csv(ALL_SUBJECT_OUTPUT_DIR / "oof_gate_metrics.csv", index=False)

with open(ALL_SUBJECT_OUTPUT_DIR / "cross_validation_summary.json", "w", encoding="utf-8") as stream:
    json.dump(all_subject_overall_metrics, stream, indent=2)

all_subject_cv_config = {
    "experiment_name": "ALL_SUBJECT_EEGNET_5FOLD_GROUPED_CV",
    "dataset": "PhysioNet EEGMMIDB v1.0.0",
    "nominal_subject_ids_attempted": ALL_SUBJECT_IDS,
    "runs": list(ALL_SUBJECT_RUNS),
    "class_mapping": EVENT_TO_LABEL,
    "class_names": CLASS_NAMES,
    "model": "Braindecode EEGNet via EEGClassifier",
    "cross_validation": "StratifiedGroupKFold, 5 folds; groups=subject ID; each eligible subject held out exactly once",
    "preprocessing": {
        "bandpass_hz": [LOW_CUT_HZ, HIGH_CUT_HZ],
        "epoch_seconds_after_event": [WINDOW_TMIN, WINDOW_TMAX],
        "normalization": "per-channel mean/std fitted only on that fold's training subjects",
        "channel_order": "aligned to S001 reference order when channel set exactly matches; otherwise run excluded",
    },
    "training": {
        "epochs": ALL_SUBJECT_EPOCHS,
        "batch_size": ALL_SUBJECT_BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": ALL_SUBJECT_LEARNING_RATE,
        "loss": "unweighted CrossEntropyLoss",
        "seed_base": SEED,
        "device": DEVICE,
    },
    "gate_thresholds_diagnostic_only": list(ALL_SUBJECT_THRESHOLDS),
    "file_hashes_computed": bool(ALL_SUBJECT_HASH_FILES),
    "metrics": all_subject_overall_metrics,
    "note": "Every eligible subject is tested once out of fold and is in the training group for other folds. No subject overlaps train/test within a fold. Test folds were not used to tune the confidence threshold.",
}
with open(ALL_SUBJECT_OUTPUT_DIR / "experiment_config_and_results.json", "w", encoding="utf-8") as stream:
    json.dump(all_subject_cv_config, stream, indent=2)

print("ALL-SUBJECT OUT-OF-FOLD RESULTS")
print(json.dumps(all_subject_overall_metrics, indent=2))
print("Fold metrics:")
display(fold_metrics_df.round(4))
print("Pooled out-of-fold class metrics:")
display(class_metrics_all_subjects_df.round(4))
print("Out-of-fold majority baseline accuracy:", round(all_subject_overall_metrics["majority_baseline_oof_accuracy"], 4))
print("Pooled out-of-fold confusion matrix:")
display(all_subject_confusion)
print("Per-subject results (first 20 rows):")
display(subject_metrics_all_subjects_df.head(20).round(4))
print("Fixed confidence-gate diagnostics:")
display(gate_metrics_all_subjects_df.round(4))
print("Subject fold manifest (each eligible subject appears once as held-out test):")
display(fold_subject_manifest_df.head(20))
print("Fold-by-fold train/test subject assignment audit (first 30 rows):")
display(fold_assignment_manifest_df.head(30))
print("Train/test subject overlap by fold:")
overlap_by_fold = {}
for fold_id, assignment_frame in fold_assignment_manifest_df.groupby("fold"):
    train_ids = set(assignment_frame.loc[assignment_frame["role"] == "TRAIN", "subject"])
    test_ids = set(assignment_frame.loc[assignment_frame["role"] == "TEST", "subject"])
    overlap_by_fold[int(fold_id)] = len(train_ids & test_ids)
print(overlap_by_fold)
print("Saved all-subject artifacts to:", ALL_SUBJECT_OUTPUT_DIR.resolve())


Fold 1/5: train=84 subjects / 7560 epochs; test=22 subjects / 1975 epochs; accuracy=0.540, macro-F1=0.451, training=13.0s
Fold 2/5: train=85 subjects / 7645 epochs; test=21 subjects / 1890 epochs; accuracy=0.540, macro-F1=0.403, training=13.0s
Fold 3/5: train=85 subjects / 7645 epochs; test=21 subjects / 1890 epochs; accuracy=0.562, macro-F1=0.486, training=13.2s
Fold 4/5: train=85 subjects / 7645 epochs; test=21 subjects / 1890 epochs; accuracy=0.541, macro-F1=0.420, training=13.1s
Fold 5/5: train=85 subjects / 7645 epochs; test=21 subjects / 1890 epochs; accuracy=0.550, macro-F1=0.444, training=12.9s
ALL-SUBJECT OUT-OF-FOLD RESULTS
{
  "experiment": "All-subject EEGNet \u2014 5-fold subject-independent CV",
  "eligible_subjects": 106,
  "excluded_subjects": 3,
  "runs_permitted": [
    "R04",
    "R08",
    "R12"
  ],
  "total_tested_epochs_out_of_fold": 9535,
  "accuracy": 0.5466177241740955,
  "balanced_accuracy": 0.44755640037507866,
  "macro_f1": 0.4446665041878888,
  "majority_b

,fold,train_subjects,test_subjects,train_epochs,test_epochs,training_seconds,accuracy,balanced_accuracy,macro_f1,majority_baseline_accuracy,train_test_subject_overlap
0,1,84,22,7560,1975,13.0422,0.5403,0.4510,0.4506,0.5003,0
1,2,85,21,7645,1890,12.9774,0.5402,0.4197,0.4028,0.5000,0
2,3,85,21,7645,1890,13.2162,0.5619,0.4846,0.4861,0.5000,0
3,4,85,21,7645,1890,13.1213,0.5413,0.4346,0.4202,0.5000,0
4,5,85,21,7645,1890,12.9355,0.5497,0.4478,0.4439,0.5000,0


Pooled out-of-fold class metrics:


,class,support,precision,recall,f1
0,Rest,4768,0.5776,0.8440,0.6858
1,Left Hand,2405,0.4346,0.2337,0.3039
2,Right Hand,2362,0.4910,0.2650,0.3442


Out-of-fold majority baseline accuracy: 0.5001
Pooled out-of-fold confusion matrix:


,pred_Rest,pred_Left Hand,pred_Right Hand
true_Rest,4024,412,332
true_Left Hand,1526,562,317
true_Right Hand,1417,319,626


... (5 outputs left out)

,subject,held_out_test_fold,included_in_training_folds
0,S001,4,"1,2,3,5"
1,S002,5,"1,2,3,4"
2,S003,2,"1,3,4,5"
3,S004,5,"1,2,3,4"
4,S005,1,"2,3,4,5"
5,S006,2,"1,3,4,5"
6,S007,5,"1,2,3,4"
7,S008,3,"1,2,4,5"
8,S009,2,"1,3,4,5"
9,S010,1,"2,3,4,5"


Fold-by-fold train/test subject assignment audit (first 30 rows):


,fold,subject,role
0,1,S001,TRAIN
1,1,S002,TRAIN
2,1,S003,TRAIN
3,1,S004,TRAIN
4,1,S006,TRAIN
5,1,S007,TRAIN
6,1,S008,TRAIN
7,1,S009,TRAIN
8,1,S011,TRAIN
9,1,S012,TRAIN


Train/test subject overlap by fold:
{1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
Saved all-subject artifacts to: C:\Users\cdsch\OneDrive\Documents\AIS\NeuroMotion\outputs\all_subject_cross_validation


### Interpreting the all-subject experiment

Use the **pooled out-of-fold** metrics as the primary all-subject result because every eligible participant is predicted only by a model that did not train on that participant. Report the fold mean and standard deviation as a stability check, and inspect `oof_subject_metrics.csv` for participant-level variation. Report the number of eligible and excluded participants and the exact reason for each exclusion from `all_subject_summary.csv` and `all_subject_run_quality.csv`.

Do not treat the sum of all windows as independent people; windows are nested within subjects. The five-fold grouped procedure prevents subject leakage, but each participant still contributes multiple correlated windows. State that this is a dataset-specific cross-subject estimate, not evidence of generalization to every user or recording setup.

A model with acceptable raw accuracy can still be unusable if it mostly predicts REST or the gate withholds nearly everything. Use `oof_gate_metrics.csv` to report both the accepted output rate and **correct movement commands per all held-out epochs**. A zero movement-command denominator must be reported as undefined, not as a 0% error rate.

The run-level S001 experiment and the all-subject grouped experiment answer different questions and must be shown as separate rows/tables in the report. The all-subject CV cell trains five separate fold models; it does not overwrite or silently replace the original S001 artifacts.


**Dataset reference for the all-subject extension:** Schalk, G. (2009). *EEG Motor Movement/Imagery Dataset (version 1.0.0).* PhysioNet. https://physionet.org/content/eegmmidb/1.0.0/ (DOI: 10.13026/C28G6P). The dataset documentation describes 109 volunteers, 64 EEG channels sampled at 160 Hz, and run-dependent meanings of T0/T1/T2. The R04/R08/R12 mapping used here is the unilateral imagined-fist task mapping (T0 rest, T1 left fist, T2 right fist).